# Dynamic statistical demography for Chile with a finite-element PDE 🇨🇱

This notebook constructs a **two-sex McKendrick–von Foerster model** from actual public Chilean data. It does four things jointly:

1. treats published age groups as **integral observations**, not point values;
2. estimates positive continuous-age fertility and mortality schedules;
3. propagates population density with an actual P1 finite-element solver;
4. fits dynamic statistical factors, including a Bayesian NumPyro implementation.

No synthetic observations are used. The raw INE and Censo 2024 workbooks and the curated extracts are bundled in the repository.

## 1. Mathematical object

For sex $s\in\{m,f\}$, age $a$, and calendar time $t$, let $\color{cyan}{n_s(a,t)}$ be population density. The structural PDE is

$$
\frac{\partial \color{cyan}{n_s}}{\partial t}
+
\frac{\partial \color{cyan}{n_s}}{\partial a}
=
-\color{orange}{\mu_s(a,t)}\color{cyan}{n_s(a,t)}
+
\color{yellow}{\nu_s(a,t)}.
$$

People enter at age zero through the nonlocal fertility boundary

$$
\color{cyan}{n_s(0,t)}
=
\color{lime}{p_s(t)}
\int_0^{a_{\max}}
\color{yellow}{f(a,t)}\color{cyan}{n_f(a,t)}\,da.
$$

The crucial observation equation for a published age bin $A_i$ and time period $T_j$ is

$$
\color{white}{D_{ijs}}
\sim
\operatorname{Poisson}\!\left(
\int_{T_j}\int_{A_i}
\color{orange}{\mu_s(a,t)}\color{cyan}{n_s(a,t)}\,da\,dt
\right),
$$

with the analogous expression for births using $\color{yellow}{f(a,t)}\color{cyan}{n_f(a,t)}$. Thus the spreadsheet cells are noisy observations of **integrated event intensity**.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from chile_demographic_pde.calibration.pde_constrained import PDEConstrainedCalibrator
from chile_demographic_pde.config import ModelConfig
from chile_demographic_pde.data.age_groups import (
    aggregate_deaths_to_population_bins,
    censo_age_bins,
    fertility_age_bins,
)
from chile_demographic_pde.data.loaders import load_project_data
from chile_demographic_pde.data.observation import p1_bin_integral_operator
from chile_demographic_pde.diagnostics.forecast import rolling_origin_backtest
from chile_demographic_pde.diagnostics.interventions import (
    fertility_boost,
    mortality_reduction,
    simulate_scenario,
)
from chile_demographic_pde.diagnostics.residuals import diagnostic_report
from chile_demographic_pde.dynamics.bayesian import fit_bayesian_local_linear_trend

ROOT = Path.cwd()
DATA = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"
REPORTS = ROOT / "reports"
FIGURES.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

CYAN = "#4DEEEA"
YELLOW = "#FFE66D"
ORANGE = "#FF9F1C"
LIME = "#B8F2A1"
RED = "#FF6B6B"
WHITE = "#F5F7FA"

plt.style.use("dark_background")
plt.rcParams.update({"figure.figsize": (10, 5.5), "axes.grid": True, "grid.alpha": 0.18})
pd.set_option("display.max_columns", 20)

## 2. Actual data and validation

The model combines:

- official monthly INE provisional births and deaths, January 2025–April 2026;
- the April 2026 INE age profile for births and deaths;
- Censo 2024 population by five-year age and sex.

The Censo stock is an exposure anchor, not an exact 2026 person-year exposure. This approximation is explicit and is one of the main places where better official data would improve inference.

In [ ]:
data = load_project_data(DATA)
summary = pd.DataFrame(
    {
        "quantity": [
            "April births with specified maternal-age group",
            "April deaths",
            "Censo 2024 population",
            "monthly observations",
        ],
        "value": [
            int(data.births_age["births_total"].sum()),
            int(data.deaths_age["deaths_total"].sum()),
            int(data.population_age["population_total"].sum()),
            len(data.monthly),
        ],
    }
)
display(summary)
display(data.monthly[["period", "births_total", "deaths_total"]])

In [ ]:
fig, ax = plt.subplots()
ax.bar(
    data.births_age["age_group"],
    data.births_age["births_total"],
    color=YELLOW,
    alpha=0.9,
)
ax.set_title("INE April 2026 births by maternal-age bin")
ax.set_xlabel("Maternal-age group")
ax.set_ylabel("Registered births")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIGURES / "births_age_bins.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
ax.plot(
    data.deaths_age["age_group"],
    data.deaths_age["deaths_male"],
    marker="o",
    color=CYAN,
    label="Male",
)
ax.plot(
    data.deaths_age["age_group"],
    data.deaths_age["deaths_female"],
    marker="o",
    color=ORANGE,
    label="Female",
)
ax.set_title("INE April 2026 deaths by age and sex")
ax.set_xlabel("Age group")
ax.set_ylabel("Registered deaths")
ax.tick_params(axis="x", rotation=55)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "deaths_age_sex.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Recovering a continuous positive rate from grouped observations

The continuous log rate is represented by cubic B-splines,

$$
\log \color{yellow}{r(a)}
=
\sum_{k=1}^{K}\theta_k B_k(a).
$$

For age-bin exposure density $e(a)$, the exact model mean is approximated by high-order quadrature:

$$
\lambda_i(\theta)
=
\int_{A_i}
\color{cyan}{e(a)}
\exp\!\left(\sum_k\theta_kB_k(a)\right)da.
$$

We minimize Poisson negative log likelihood plus a second-difference penalty on spline coefficients. This is a composite-link inverse problem: smoothness chooses one plausible within-bin curve among infinitely many curves sharing the same aggregates.

The Censo exposure data end in a broad $85+$ group, which cannot identify the extreme-age hazard shape. For mortality we therefore impose the transparent tail restriction

$$
\frac{\partial}{\partial a}\log \color{orange}{\mu_s(a)}\geq 0,
\qquad a\geq40.
$$

SciPy enforces this as linear inequalities on the B-spline derivative. The restriction prevents an artificial old-age decline; it is regularization, not newly observed single-age information.

## 4. Finite-element weak form

With implicit Euler and test function $v$, the stabilized weak step is

$$
(v,n^{k+1})
+\Delta t(v,\partial_a n^{k+1})
+\Delta t(v,\mu n^{k+1})
+\Delta t\,\varepsilon(\partial_av,\partial_an^{k+1})
=(v,n^k)+\Delta t(v,\nu).
$$

After P1 assembly with `scikit-fem`:

$$
\left[
\color{cyan}{M}
+\Delta t\left(
\color{yellow}{C}
+\color{orange}{R(\mu)}
+\varepsilon\color{lime}{K}
\right)
\right]\mathbf n^{k+1}
=
\color{cyan}{M}\mathbf n^k+\Delta t\color{cyan}{M}\boldsymbol\nu.
$$

The code uses an exact age-zero Dirichlet inflow, a natural old-age outflow, sparse SciPy solves, and mild artificial diffusion to stabilize central Galerkin advection.

In [ ]:
config = ModelConfig(
    age_nodes=121,
    fertility_spline_df=8,
    mortality_spline_df=10,
    mortality_monotone_from=40.0,
    spline_penalty=20.0,
    dynamic_penalty=30.0,
    artificial_diffusion=0.015,
)
calibrator = PDEConstrainedCalibrator(
    config,
    max_iterations=6,
    tolerance=1e-4,
    damping=0.8,
)
calibration = calibrator.fit(data)
reference = calibration.reference

print(f"Iterations: {calibration.iterations}")
print(f"Converged: {calibration.converged}")
print("Maximum log-scale changes:", calibration.convergence_history)

## 5. Censo 2024 observed age-bin counts and resident-population state

Censo five-year totals, including the open $85+$ group, are **enumerated grouped counts**. They are not observed single-age data and are not resident-population estimates. The bars below are therefore the primary observation.

The dashed curve is only a descriptive, nonnegative minimum-curvature P1 interpolation. Its bin integrals equal every official grouped total to numerical tolerance; smoothness selects among exact-bin solutions rather than relaxing the observation equations. Because the bundled files do not include an official single-age resident-population series, the reference pipeline uses this curve as a transparent provisional state/exposure anchor, not as an official resident estimate.

The diagnostic table separates a dimensionless same-grid `local_curvature_indicator` from an actual coarse-versus-refined reconstruction comparison. The latter refines every age interval by a factor of two, interpolates the refined exact-bin curve onto the coarse nodes, and reports absolute and relative $L^\infty$ differences. The within-bin curve can move as the age grid changes; the grouped counts do not.

For visualization only, the open $85+$ total is divided by the explicit model-domain interval $[85,105]$ to form the last count-density bar. The official grouped count remains authoritative; age 105 is a disclosed PDE truncation, not a finite endpoint asserted by Censo.

In [ ]:
age = reference.age
population_bins = censo_age_bins(data.population_age)
bin_lower = np.array([age_bin.lower for age_bin in population_bins])
bin_upper = np.array(
    [age_bin.resolved_upper(config.maximum_age) for age_bin in population_bins]
)
bin_widths = bin_upper - bin_lower
population_total = data.population_age["population_total"].to_numpy(dtype=float)
observed_count_density = population_total / bin_widths
descriptive_total_density = reference.initial_male + reference.initial_female

fig, ax = plt.subplots()
ax.bar(
    bin_lower,
    observed_count_density,
    width=bin_widths,
    align="edge",
    color=CYAN,
    edgecolor=WHITE,
    linewidth=1.2,
    alpha=0.55,
    label="Official Censo model-domain count density",
)
ax.plot(
    age,
    descriptive_total_density,
    color=YELLOW,
    linestyle="--",
    linewidth=2.2,
    label="Descriptive exact-bin overlay — not observed single-age data",
)
ax.set_title("Censo 2024 observed age-bin counts and resident-population state")
ax.set_xlabel("Age")
ax.set_ylabel("Censo-enumerated people per year of age")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "continuous_population_density.png", dpi=160, bbox_inches="tight")
plt.show()

male_reconstruction_diagnostics = reference.censo_reconstruction_male.diagnostics
female_reconstruction_diagnostics = reference.censo_reconstruction_female.diagnostics
male_grid_comparison = reference.censo_grid_comparison_male
female_grid_comparison = reference.censo_grid_comparison_female
censo_reconstruction_diagnostics = pd.DataFrame(
    [
        {
            "sex": "male",
            "max_absolute_bin_error_people": (
                male_reconstruction_diagnostics.max_absolute_bin_error
            ),
            "max_relative_bin_error": (
                male_reconstruction_diagnostics.max_relative_bin_error
            ),
            "total_residual_people": male_reconstruction_diagnostics.total_residual,
            "constraint_condition_number": (
                male_reconstruction_diagnostics.constraint_condition_number
            ),
            "active_lower_bounds": male_reconstruction_diagnostics.active_lower_bounds,
            "minimum_density": male_reconstruction_diagnostics.minimum_density,
            "local_curvature_indicator": (
                male_reconstruction_diagnostics.local_curvature_indicator
            ),
            "coarse_node_count": male_grid_comparison.coarse_node_count,
            "refined_node_count": male_grid_comparison.refined_node_count,
            "max_absolute_density_difference": (
                male_grid_comparison.max_absolute_density_difference
            ),
            "relative_linf_density_difference": (
                male_grid_comparison.relative_linf_density_difference
            ),
        },
        {
            "sex": "female",
            "max_absolute_bin_error_people": (
                female_reconstruction_diagnostics.max_absolute_bin_error
            ),
            "max_relative_bin_error": (
                female_reconstruction_diagnostics.max_relative_bin_error
            ),
            "total_residual_people": female_reconstruction_diagnostics.total_residual,
            "constraint_condition_number": (
                female_reconstruction_diagnostics.constraint_condition_number
            ),
            "active_lower_bounds": female_reconstruction_diagnostics.active_lower_bounds,
            "minimum_density": female_reconstruction_diagnostics.minimum_density,
            "local_curvature_indicator": (
                female_reconstruction_diagnostics.local_curvature_indicator
            ),
            "coarse_node_count": female_grid_comparison.coarse_node_count,
            "refined_node_count": female_grid_comparison.refined_node_count,
            "max_absolute_density_difference": (
                female_grid_comparison.max_absolute_density_difference
            ),
            "relative_linf_density_difference": (
                female_grid_comparison.relative_linf_density_difference
            ),
        },
    ]
)
display(censo_reconstruction_diagnostics)

fig, ax = plt.subplots()
ax.plot(age, reference.fertility.rate(age), color=YELLOW, linewidth=2.5)
ax.set_xlim(10, 55)
ax.set_title("Estimated continuous maternal-age fertility schedule")
ax.set_xlabel("Maternal age")
ax.set_ylabel("Annual births per woman-year")
fig.tight_layout()
fig.savefig(FIGURES / "fertility_rate_age.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
ax.semilogy(age, reference.mortality_male.rate(age), color=CYAN, label="Male")
ax.semilogy(age, reference.mortality_female.rate(age), color=ORANGE, label="Female")
ax.set_title("Estimated continuous mortality hazards")
ax.set_xlabel("Age")
ax.set_ylabel("Annual mortality intensity — logarithmic scale")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "mortality_hazard_age.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Compare the observed bin counts with integrals of the fitted continuous intensities.
fertility_bins, births_selected = fertility_age_bins(data.births_age)
fertility_operator = p1_bin_integral_operator(age, fertility_bins)
fertility_expected = fertility_operator.matrix.values @ (
    reference.fertility.rate(age) * reference.initial_female
) * config.time_step_years

population_bins = censo_age_bins(data.population_age)
mortality_operator = p1_bin_integral_operator(
    age,
    population_bins,
    open_bin_end=config.maximum_age,
)
mortality_expected = mortality_operator.matrix.values @ (
    reference.mortality_male.rate(age) * reference.initial_male
    + reference.mortality_female.rate(age) * reference.initial_female
) * config.time_step_years
mortality_observed = aggregate_deaths_to_population_bins(
    data.deaths_age, population_bins, "deaths_total", config.maximum_age
)

bin_fit = pd.DataFrame(
    {
        "fertility_age_bin": births_selected["age_group"],
        "births_observed": births_selected["births_total"].to_numpy(),
        "births_integrated_model": np.round(fertility_expected, 1),
    }
)
display(bin_fit)

fig, ax = plt.subplots()
x = np.arange(len(population_bins))
ax.plot(x, mortality_observed, marker="o", color=WHITE, label="Observed")
ax.plot(x, mortality_expected, marker="s", color=ORANGE, label="Integrated model")
ax.set_xticks(x, [b.label for b in population_bins], rotation=55, ha="right")
ax.set_title("Mortality: observed grouped counts versus continuous-rate integrals")
ax.set_ylabel("April 2026 deaths")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "mortality_bin_fit.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Dynamic PDE-constrained calibration

Only April 2026 supplies the detailed age profile in the bundled data. Therefore the identified dynamic specification is

$$
\color{yellow}{f(a,t)}=\color{yellow}{f_0(a)}e^{\eta_f(t)},
\qquad
\color{orange}{\mu_s(a,t)}=\color{orange}{\mu_{0,s}(a)}e^{\eta_\mu(t)}.
$$

The fitting loop alternates between:

1. FEM propagation of $n_s(a,t)$;
2. computation of PDE-implied unit-rate count offsets;
3. penalized Poisson estimation of $\eta_f(t)$ and $\eta_\mu(t)$;
4. log-scale damping and repetition.

Thus changes in population composition feed back into the statistical likelihood.

In [ ]:
periods = pd.to_datetime(data.monthly["period"])
observed_births = data.monthly["births_total"].to_numpy(dtype=float)
observed_deaths = data.monthly["deaths_total"].to_numpy(dtype=float)
predicted_births = calibration.simulation.predicted_births
predicted_deaths = calibration.simulation.predicted_deaths

fig, ax = plt.subplots()
ax.plot(periods, observed_births, marker="o", color=WHITE, label="INE observed")
ax.plot(periods, predicted_births, marker="s", color=YELLOW, label="PDE-constrained fit")
ax.set_title("Monthly births: observed versus PDE-constrained model")
ax.set_ylabel("Births")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "monthly_birth_fit.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
ax.plot(periods, observed_deaths, marker="o", color=WHITE, label="INE observed")
ax.plot(periods, predicted_deaths, marker="s", color=ORANGE, label="PDE-constrained fit")
ax.set_title("Monthly deaths: observed versus PDE-constrained model")
ax.set_ylabel("Deaths")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "monthly_death_fit.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
ax.semilogy(
    np.arange(1, calibration.iterations + 1),
    calibration.convergence_history,
    marker="o",
    color=LIME,
)
ax.axhline(1e-4, color=RED, linestyle="--", label="Tolerance")
ax.set_title("PDE/statistical block-coordinate convergence")
ax.set_xlabel("Iteration")
ax.set_ylabel("Maximum absolute log-multiplier update")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "calibration_convergence.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Diagnostics and honest forecast evaluation

The fitted monthly factors are highly flexible relative to only 16 observations, so in-sample residuals are expected to be small. Pearson dispersion is therefore descriptive rather than a classical degrees-of-freedom-adjusted estimate. The Ljung–Box test is useful as a warning about remaining temporal structure but has low power and unstable calibration in such a short series.

In [ ]:
birth_diagnostics = diagnostic_report(
    observed_births, predicted_births, lags=4, estimated_parameters=4
)
death_diagnostics = diagnostic_report(
    observed_deaths, predicted_deaths, lags=4, estimated_parameters=4
)

diagnostics_table = pd.DataFrame(
    {
        "metric": [
            "Poisson deviance",
            "Pearson dispersion — descriptive",
            "MAE",
            "RMSE",
            "Ljung–Box statistic",
            "Ljung–Box p-value",
        ],
        "births": [
            birth_diagnostics.deviance,
            birth_diagnostics.pearson_dispersion,
            birth_diagnostics.mean_absolute_error,
            birth_diagnostics.root_mean_squared_error,
            birth_diagnostics.ljung_box_statistic,
            birth_diagnostics.ljung_box_pvalue,
        ],
        "deaths": [
            death_diagnostics.deviance,
            death_diagnostics.pearson_dispersion,
            death_diagnostics.mean_absolute_error,
            death_diagnostics.root_mean_squared_error,
            death_diagnostics.ljung_box_statistic,
            death_diagnostics.ljung_box_pvalue,
        ],
    }
)
display(diagnostics_table.round(4))

fig, ax = plt.subplots()
ax.axhline(0.0, color=WHITE, linewidth=1)
ax.plot(periods, birth_diagnostics.pearson_residuals, marker="o", color=YELLOW, label="Births")
ax.plot(periods, death_diagnostics.pearson_residuals, marker="s", color=ORANGE, label="Deaths")
ax.set_title("Pearson residuals")
ax.set_ylabel("Standardized residual")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "pearson_residuals.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
birth_backtest = rolling_origin_backtest(
    observed_births, minimum_training=8, penalty=config.dynamic_penalty
)
death_backtest = rolling_origin_backtest(
    observed_deaths, minimum_training=8, penalty=config.dynamic_penalty
)
birth_backtest["series"] = "births"
death_backtest["series"] = "deaths"
backtest = pd.concat([birth_backtest, death_backtest], ignore_index=True)
backtest.to_csv(REPORTS / "rolling_origin_backtest.csv", index=False)

display(
    backtest.groupby("series")["error"]
    .agg(MAE=lambda x: np.mean(np.abs(x)), RMSE=lambda x: np.sqrt(np.mean(x**2)))
    .round(2)
)

fig, ax = plt.subplots()
for frame, color, label in [
    (birth_backtest, YELLOW, "Birth forecast errors"),
    (death_backtest, ORANGE, "Death forecast errors"),
]:
    ax.plot(frame["origin"], frame["error"], marker="o", color=color, label=label)
ax.axhline(0.0, color=WHITE, linewidth=1)
ax.set_title("One-step rolling-origin errors — no future leakage")
ax.set_xlabel("Forecasted observation index")
ax.set_ylabel("Actual minus forecast")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "rolling_origin_errors.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Bayesian dynamic implementation with NumPyro

The Bayesian component places a damped local-linear prior on the latent log multiplier:

$$
b_t=\rho b_{t-1}+\sigma_\eta z_t,
\qquad
\eta_t=\eta_{t-1}+b_t,
\qquad
z_t\sim\mathcal N(0,1),
$$

$$
Y_t\sim\operatorname{Poisson}\left(O_te^{\eta_t}\right),
\qquad \rho=0.8.
$$

where $O_t$ is the unit-rate count implied by the current FEM population path. The notebook uses NumPyro `AutoNormal` stochastic variational inference for speed and portability; the package also exposes a slower `inference_method="nuts"` option.

Mean-field SVI tends to underestimate posterior dependence and sometimes interval width. It is a Bayesian approximation, not a substitute for checking NUTS on a production model.

In [ ]:
# Linearity in the period multiplier lets us recover the current PDE unit-rate offset.
birth_unit_offset = predicted_births / calibration.fertility_scale

bayesian_births = fit_bayesian_local_linear_trend(
    observed_births,
    offset=birth_unit_offset,
    inference_method="svi",
    svi_steps=4_000,
    num_samples=1_000,
    random_seed=7,
    trend_damping=0.8,
)

bayesian_fit_table = pd.DataFrame(
    {
        "period": data.monthly["period"],
        "observed": observed_births,
        "posterior_mean": bayesian_births.posterior_mean,
        "posterior_lower": bayesian_births.posterior_lower,
        "posterior_upper": bayesian_births.posterior_upper,
    }
)
display(bayesian_fit_table.round(1))
print(f"Final SVI ELBO loss: {bayesian_births.final_loss:,.2f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(periods, observed_births, marker="o", color=WHITE, label="Observed")
ax.plot(periods, bayesian_births.posterior_mean, color=CYAN, label="Bayesian posterior mean")
ax.fill_between(
    periods,
    bayesian_births.posterior_lower,
    bayesian_births.posterior_upper,
    color=CYAN,
    alpha=0.20,
    label="95% variational interval",
)
ax.set_title("Bayesian Poisson structural trend for births")
ax.set_ylabel("Births")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "bayesian_birth_fit.png", dpi=160, bbox_inches="tight")
plt.show()

future_horizon = 6
scale_forecast = bayesian_births.forecast(future_horizon, random_seed=23)
future_offset = np.repeat(birth_unit_offset[-1], future_horizon)
future_periods = pd.date_range(periods.iloc[-1] + pd.offsets.MonthBegin(), periods=future_horizon, freq="MS")
forecast_table = pd.DataFrame(
    {
        "period": future_periods.strftime("%Y-%m"),
        "posterior_median": scale_forecast.median * future_offset,
        "posterior_mean": scale_forecast.mean * future_offset,
        "lower_95": scale_forecast.lower * future_offset,
        "upper_95": scale_forecast.upper * future_offset,
    }
)
display(forecast_table.round(0))

fig, ax = plt.subplots()
ax.plot(periods, observed_births, marker="o", color=WHITE, label="Observed")
ax.plot(future_periods, forecast_table["posterior_median"], marker="s", color=LIME, label="Forecast median")
ax.fill_between(
    future_periods,
    forecast_table["lower_95"],
    forecast_table["upper_95"],
    color=LIME,
    alpha=0.18,
    label="95% variational interval",
)
ax.set_title("Six-month Bayesian birth forecast — conditional on latest PDE exposure")
ax.set_ylabel("Births")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "bayesian_birth_forecast.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Structural interventions propagated through the PDE

These are **conditional scenarios**, not causal estimates:

- baseline: latest fitted rate schedules held fixed;
- fertility intervention: fertility multiplied by $1.10$ from ages 25 to 34;
- mortality intervention: mortality multiplied by $0.90$ from age 65 onward.

Each intervention changes future births or deaths, which then changes the population state and future exposures through the PDE.

In [ ]:
baseline = simulate_scenario(calibration, years=10.0)
higher_fertility = simulate_scenario(
    calibration,
    years=10.0,
    fertility_multiplier=fertility_boost(25.0, 35.0, 1.10),
)
lower_old_age_mortality = simulate_scenario(
    calibration,
    years=10.0,
    mortality_multiplier=mortality_reduction(65.0, 105.0, 0.90),
)

scenario_table = pd.DataFrame(
    {
        "scenario": ["baseline", "+10% fertility ages 25–34", "−10% mortality ages 65+"],
        "population_after_10y": [
            baseline.population_total[-1],
            higher_fertility.population_total[-1],
            lower_old_age_mortality.population_total[-1],
        ],
        "cumulative_births": [
            baseline.simulation.predicted_births.sum(),
            higher_fertility.simulation.predicted_births.sum(),
            lower_old_age_mortality.simulation.predicted_births.sum(),
        ],
        "cumulative_deaths": [
            baseline.simulation.predicted_deaths.sum(),
            higher_fertility.simulation.predicted_deaths.sum(),
            lower_old_age_mortality.simulation.predicted_deaths.sum(),
        ],
    }
)
scenario_table.to_csv(REPORTS / "intervention_summary.csv", index=False)
display(scenario_table.round(0))

months = np.arange(baseline.population_total.size) / 12.0
fig, ax = plt.subplots()
ax.plot(months, baseline.population_total, color=WHITE, label="Baseline")
ax.plot(months, higher_fertility.population_total, color=YELLOW, label="Higher fertility")
ax.plot(months, lower_old_age_mortality.population_total, color=LIME, label="Lower 65+ mortality")
ax.set_title("Conditional 10-year population scenarios — zero age-specific migration")
ax.set_xlabel("Years after April 2026")
ax.set_ylabel("Population represented on the FEM age domain")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "population_interventions.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
years = np.arange(1, baseline.annual_births.size + 1)
ax.plot(years, baseline.annual_births, color=WHITE, label="Baseline births")
ax.plot(years, higher_fertility.annual_births, color=YELLOW, label="Fertility intervention births")
ax.plot(years, baseline.annual_deaths, color=ORANGE, label="Baseline deaths")
ax.plot(years, lower_old_age_mortality.annual_deaths, color=LIME, label="Mortality intervention deaths")
ax.set_title("Annual event flows generated by the structural PDE")
ax.set_xlabel("Scenario year")
ax.set_ylabel("Events")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "annual_intervention_flows.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. What the results do—and do not—say

### Supported conclusions

- Treating age bins as integrals matters mathematically: the fitted object is a continuous intensity, while the spreadsheet contains linear functionals of that intensity.
- The nondecreasing mortality-tail constraint prevents the broad $85+$ exposure group from creating an unsupported extreme-age hazard reversal.
- The P1 FEM creates the expected cohort ageing along $a-t=\text{constant}$ and incorporates mortality as a reaction term and fertility as an endogenous inflow boundary.
- The iterative calibration converges quickly because demographic composition changes only moderately over 16 months.
- Monthly births and deaths contain visible short-run structure that a constant-rate PDE would miss.
- Age-targeted interventions act through both event flows and future exposure distributions.

### Unsupported conclusions

- The model does not identify arbitrary age-by-time interactions from one detailed age profile.
- The 10-year paths are not official projections because migration is set to zero and future time factors are held fixed.
- The intervention effects are not causal estimates.
- The Bayesian intervals are variational and should be checked against longer NUTS runs before high-stakes use.

In [ ]:
model_summary = {
    "data": {
        "monthly_periods": len(data.monthly),
        "censo_population": int(data.population_age["population_total"].sum()),
        "april_2026_births": int(data.monthly.iloc[-1]["births_total"]),
        "april_2026_deaths": int(data.monthly.iloc[-1]["deaths_total"]),
    },
    "calibration": {
        "iterations": calibration.iterations,
        "converged": calibration.converged,
        "birth_deviance": birth_diagnostics.deviance,
        "death_deviance": death_diagnostics.deviance,
        "birth_rolling_mae": float(np.mean(np.abs(birth_backtest["error"]))),
        "death_rolling_mae": float(np.mean(np.abs(death_backtest["error"]))),
    },
    "bayesian_births": {
        "method": bayesian_births.inference_method,
        "final_loss": bayesian_births.final_loss,
    },
    "scenarios": scenario_table.set_index("scenario").to_dict(orient="index"),
}
(REPORTS / "model_summary.json").write_text(json.dumps(model_summary, indent=2), encoding="utf-8")
model_summary

## 11. Why these libraries

| Component | Selected library | Why |
|---|---|---|
| Finite elements | `scikit-fem` | Real variational assembly with a lightweight pure-Python and sparse-SciPy stack |
| Deterministic inference | SciPy | Stable optimization, B-splines, quadrature, sparse linear algebra |
| Bayesian inference | NumPyro + JAX | Composable structural models, SVI and optional NUTS |
| Diagnostics | statsmodels | Established Ljung–Box implementation |
| Data | pandas | Clear tabular transformations after spreadsheet extraction |
| Configuration | Pydantic | Immutable validation and explicit numerical settings |
| Reproducibility | uv | Locked cross-platform environment and concise commands |

`FEniCSx` and Firedrake would be stronger for large multidimensional PDE systems, but they impose a much heavier native/MPI stack. FiPy is an attractive conservative finite-volume benchmark but does not answer the request for FEM. JAX-FEM is promising for end-to-end differentiability, but the current sparse/autodiff ecosystem requires more bespoke numerical engineering than this transparent 1D model warrants.

See `docs/LIBRARY_COMPARISON.md` for a fuller comparison and `docs/LIMITATIONS.md` for the next data/model extensions.